In [ ]:
import shutil
import os

src = "/kaggle/input/datasets/quanghijr/outputs"
dst = "/kaggle/working/outputs"

if os.path.exists(dst):
    shutil.rmtree(dst)

# Copy toàn bộ thư mục
shutil.copytree(src, dst)

print("Copy completed!")

# 1. Config

In [ ]:
cfg_mobilenet_v2 = {
    "name": "mobilenet_v2",
    # anchor
    "min_sizes": [[8, 16], [32, 64], [128, 256]],
    "steps": [8, 16, 32],
    "variance": [0.1, 0.2],
    "clip": False,
    # loss
    "loc_weight": 2.0,
    # training
    "batch_size": 32,
    "epochs": 250,
    "milestones": [190, 220],
    # input
    "image_size": 640,
    # backbone
    "pretrain": True,
    "return_layers": [6, 13, 18],
    "in_channel": 32,
    "out_channel": 128,
}


In [ ]:
from copy import deepcopy

_CONFIGS = {
    "mobilenet_v2": cfg_mobilenet_v2,
}


def get_config(name: str):
    """
    Get configuration by model name.

    Args:
        name (str): Name of the backbone/model.

    Returns:
        dict: A deep copy of the configuration.

    Raises:
        ValueError: If config name is not found.
    """
    if name not in _CONFIGS:
        available = ", ".join(_CONFIGS.keys())
        raise ValueError(f"Unknown config '{name}'. Available: [{available}]")

    return deepcopy(_CONFIGS[name])


# 2. Box Utils

In [ ]:
from __future__ import annotations

import numpy as np
import torch
from torch import Tensor


def _reset_match_targets(
    loc_t: Tensor,
    conf_t: Tensor,
    landm_t: Tensor,
    idx: int,
    landm_valid_t: Tensor | None = None,
) -> None:
    """Fill one sample's match targets with background defaults."""
    loc_t[idx].zero_()
    conf_t[idx].zero_()
    landm_t[idx].zero_()
    if landm_valid_t is not None:
        landm_valid_t[idx].zero_()


def point_form(boxes: Tensor) -> Tensor:
    """Convert priors from (cx, cy, w, h) to (xmin, ymin, xmax, ymax)."""
    return torch.cat(
        (boxes[:, :2] - boxes[:, 2:] / 2, boxes[:, :2] + boxes[:, 2:] / 2), 1
    )


def center_size(boxes: Tensor) -> Tensor:
    """Convert boxes from (xmin, ymin, xmax, ymax) to (cx, cy, w, h)."""
    return torch.cat(
        ((boxes[:, 2:] + boxes[:, :2]) / 2, boxes[:, 2:] - boxes[:, :2]), 1
    )


def intersect(box_a: Tensor, box_b: Tensor) -> Tensor:
    """Compute pairwise intersection areas between two box collections."""
    max_xy = torch.minimum(box_a[:, None, 2:], box_b[None, :, 2:])
    min_xy = torch.maximum(box_a[:, None, :2], box_b[None, :, :2])
    inter = torch.clamp(max_xy - min_xy, min=0)
    return inter[:, :, 0] * inter[:, :, 1]


def jaccard(box_a: Tensor, box_b: Tensor) -> Tensor:
    """Compute pairwise IoU between two box collections."""
    inter = intersect(box_a, box_b)
    area_a = ((box_a[:, 2] - box_a[:, 0]) * (box_a[:, 3] - box_a[:, 1]))[:, None]
    area_b = ((box_b[:, 2] - box_b[:, 0]) * (box_b[:, 3] - box_b[:, 1]))[None, :]
    union = area_a + area_b - inter
    return inter / torch.clamp(union, min=1e-12)


def matrix_iou(box_a: np.ndarray, box_b: np.ndarray) -> np.ndarray:
    """NumPy IoU helper used by data augmentation pipelines."""
    top_left = np.maximum(box_a[:, None, :2], box_b[None, :, :2])
    bottom_right = np.minimum(box_a[:, None, 2:], box_b[None, :, 2:])
    inter = np.clip(bottom_right - top_left, a_min=0, a_max=None)
    inter_area = inter[:, :, 0] * inter[:, :, 1]

    area_a = ((box_a[:, 2] - box_a[:, 0]) * (box_a[:, 3] - box_a[:, 1]))[:, None]
    area_b = ((box_b[:, 2] - box_b[:, 0]) * (box_b[:, 3] - box_b[:, 1]))[None, :]
    union = area_a + area_b - inter_area
    return inter_area / np.clip(union, a_min=1e-12, a_max=None)


def matrix_iof(box_a: np.ndarray, box_b: np.ndarray) -> np.ndarray:
    """NumPy intersection-over-foreground helper used by random cropping."""
    top_left = np.maximum(box_a[:, None, :2], box_b[None, :, :2])
    bottom_right = np.minimum(box_a[:, None, 2:], box_b[None, :, 2:])
    inter = np.clip(bottom_right - top_left, a_min=0, a_max=None)
    inter_area = inter[:, :, 0] * inter[:, :, 1]

    area_a = ((box_a[:, 2] - box_a[:, 0]) * (box_a[:, 3] - box_a[:, 1]))[:, None]
    return inter_area / np.clip(area_a, a_min=1e-12, a_max=None)


def match(
    threshold: float,
    truths: Tensor,
    priors: Tensor,
    variances: list[float] | tuple[float, float],
    labels: Tensor,
    landms: Tensor,
    loc_t: Tensor,
    conf_t: Tensor,
    landm_t: Tensor,
    idx: int,
    landm_valid_t: Tensor | None = None,
    best_prior_iou_floor: float = 0.2,
) -> None:
    """
    Match each prior with the best ground-truth box, then encode regression targets.

    ``truths`` is expected in point-form coordinates and ``priors`` in center-size
    format. Both must be normalized to the same image scale.

    ``best_prior_iou_floor`` keeps us from force-matching a ground truth whose best
    prior is still a very poor fit. Set it to ``0.0`` if you want SSD-style forced
    matching for every GT.

    ``landm_valid_t`` can be provided to preserve which matched priors have valid
    landmark supervision. It may be shaped per-anchor, per-point, or per-coordinate.
    This is needed because datasets like WIDER FACE encode missing landmarks with
    ``-1`` coordinates.
    """
    num_priors = priors.size(0)

    if truths.numel() == 0:
        _reset_match_targets(loc_t, conf_t, landm_t, idx, landm_valid_t)
        return

    overlaps = jaccard(truths, point_form(priors))

    best_prior_overlap, best_prior_idx = overlaps.max(1, keepdim=True)
    best_truth_overlap, best_truth_idx = overlaps.max(0, keepdim=True)

    best_prior_idx = best_prior_idx.squeeze(1)
    best_prior_overlap = best_prior_overlap.squeeze(1)
    best_truth_idx = best_truth_idx.squeeze(0)
    best_truth_overlap = best_truth_overlap.squeeze(0)

    valid_gt = best_prior_overlap >= best_prior_iou_floor
    valid_gt_idx = torch.nonzero(valid_gt, as_tuple=False).squeeze(1)
    best_prior_idx = best_prior_idx[valid_gt]
    if best_prior_idx.numel() == 0:
        _reset_match_targets(loc_t, conf_t, landm_t, idx, landm_valid_t)
        return

    best_truth_overlap.index_fill_(0, best_prior_idx, 2)
    for gt_idx, prior_idx in zip(valid_gt_idx, best_prior_idx):
        best_truth_idx[prior_idx] = gt_idx

    matches = truths[best_truth_idx]
    conf = labels[best_truth_idx].clone()
    conf[best_truth_overlap < threshold] = 0

    matched_landms = landms[best_truth_idx]
    matched_landms_points = matched_landms.view(-1, 5, 2)
    point_valid = matched_landms_points.ge(0).all(dim=2)
    coord_valid = point_valid.unsqueeze(-1).expand(-1, -1, 2).reshape(-1, 10)
    anchor_valid = point_valid.all(dim=1)
    loc = encode(matches, priors, variances)
    landm = encode_landm(matched_landms, priors, variances)
    loc[conf <= 0] = 0
    landm[~((conf > 0).unsqueeze(1) & coord_valid)] = 0

    loc_t[idx] = loc
    conf_t[idx] = conf.to(conf_t.dtype)
    landm_t[idx] = landm
    if landm_valid_t is not None:
        if landm_valid_t.dim() == 2:
            landm_valid_t[idx] = (anchor_valid & (conf > 0)).to(landm_valid_t.dtype)
        elif landm_valid_t.dim() == 3 and landm_valid_t.size(-1) == 5:
            landm_valid_t[idx] = (point_valid & (conf > 0).unsqueeze(1)).to(
                landm_valid_t.dtype
            )
        elif landm_valid_t.dim() == 3 and landm_valid_t.size(-1) == 10:
            landm_valid_t[idx] = (coord_valid & (conf > 0).unsqueeze(1)).to(
                landm_valid_t.dtype
            )
        else:
            raise ValueError(
                "landm_valid_t must have shape (batch, priors), (batch, priors, 5), "
                "or (batch, priors, 10)"
            )


def encode(
    matched: Tensor, priors: Tensor, variances: list[float] | tuple[float, float]
) -> Tensor:
    """Encode matched boxes relative to prior centers and sizes."""
    centers = (matched[:, :2] + matched[:, 2:]) / 2
    g_cxcy = (centers - priors[:, :2]) / (variances[0] * priors[:, 2:])

    sizes = (matched[:, 2:] - matched[:, :2]) / priors[:, 2:]
    g_wh = torch.log(torch.clamp(sizes, min=1e-12)) / variances[1]
    return torch.cat((g_cxcy, g_wh), 1)


def encode_landm(
    matched: Tensor, priors: Tensor, variances: list[float] | tuple[float, float]
) -> Tensor:
    """Encode five facial landmarks relative to each prior."""
    matched = matched.view(-1, 5, 2)
    priors_xy = priors[:, :2].unsqueeze(1).expand(-1, 5, -1)
    priors_wh = priors[:, 2:].unsqueeze(1).expand(-1, 5, -1)

    encoded = (matched - priors_xy) / (variances[0] * priors_wh)
    return encoded.reshape(-1, 10)


def decode(
    loc: Tensor, priors: Tensor, variances: list[float] | tuple[float, float]
) -> Tensor:
    """Decode predicted box deltas back to point-form boxes."""
    boxes = torch.cat(
        (
            priors[:, :2] + loc[:, :2] * variances[0] * priors[:, 2:],
            priors[:, 2:] * torch.exp(loc[:, 2:] * variances[1]),
        ),
        1,
    )
    boxes[:, :2] -= boxes[:, 2:] / 2
    boxes[:, 2:] += boxes[:, :2]
    return boxes


def decode_landm(
    pre: Tensor, priors: Tensor, variances: list[float] | tuple[float, float]
) -> Tensor:
    """Decode predicted landmark deltas back to normalized coordinates."""
    decoded = pre.view(-1, 5, 2) * variances[0] * priors[:, 2:].unsqueeze(1)
    decoded += priors[:, :2].unsqueeze(1)
    return decoded.reshape(-1, 10)


def log_sum_exp(x: Tensor) -> Tensor:
    """Numerically stable log-sum-exp used by hard negative mining."""
    x_max = x.detach().max(dim=1, keepdim=True).values
    return torch.log(torch.sum(torch.exp(x - x_max), dim=1, keepdim=True)) + x_max

# 3. Prior Box

In [ ]:
from __future__ import annotations

from math import ceil

import torch
from torch import Tensor


class PriorBox:
    """
    Generate RetinaFace priors in ``(cx, cy, w, h)`` format.

    The generated coordinates are normalized to the input image size so they can
    be consumed directly by the box encode/decode logic.
    """

    def __init__(self, cfg: dict, image_size: tuple[int, int]) -> None:
        self.image_size = image_size

        self.min_sizes: list[list[int]] = cfg["min_sizes"]
        self.steps: list[int] = cfg["steps"]
        self.clip: bool = cfg["clip"]

        if len(self.min_sizes) != len(self.steps):
            raise ValueError(
                "cfg['min_sizes'] and cfg['steps'] must have the same length"
            )

        image_height, image_width = self.image_size
        self.feature_maps: list[list[int]] = [
            [ceil(image_height / step), ceil(image_width / step)] for step in self.steps
        ]

    def forward(self) -> Tensor:
        anchors: list[float] = []
        image_height, image_width = self.image_size

        for level_idx, feature_map in enumerate(self.feature_maps):
            min_sizes = self.min_sizes[level_idx]
            step = self.steps[level_idx]

            for row in range(feature_map[0]):
                for col in range(feature_map[1]):
                    cx = (col + 0.5) * step / image_width
                    cy = (row + 0.5) * step / image_height

                    for min_size in min_sizes:
                        scale_x = min_size / image_width
                        scale_y = min_size / image_height
                        anchors.extend([cx, cy, scale_x, scale_y])

        priors = torch.tensor(anchors, dtype=torch.float32).view(-1, 4)
        if self.clip:
            priors.clamp_(min=0.0, max=1.0)
        return priors

# 4. Transform

In [ ]:
from __future__ import annotations

import random
from typing import Any

import numpy as np
import torch
from PIL import Image, ImageEnhance
from torch import Tensor


__all__ = [
    "RetinaFaceTrainTransform",
    "RetinaFaceEvalTransform",
]


def _normalize_bgr_mean(
    bgr_mean: tuple[float, float, float] | tuple[float, ...],
) -> tuple[float, float, float]:
    if len(bgr_mean) != 3:
        raise ValueError("bgr_mean must have exactly 3 values (B, G, R)")
    return float(bgr_mean[0]), float(bgr_mean[1]), float(bgr_mean[2])


class RetinaFaceTrainTransform:
    """
    Standard RetinaFace train-time preprocessing and augmentation.

    Pipeline:
    1. photometric distortion
    2. random square crop
    3. random horizontal flip
    4. pad to square
    5. resize to ``image_size``
    6. convert RGB to BGR and subtract mean
    """

    def __init__(
        self,
        image_size: int = 640,
        bgr_mean: tuple[float, float, float] = (104.0, 117.0, 123.0),
        min_face_size: float = 0.0,
        crop_scales: tuple[float, ...] = (0.3, 0.45, 0.6, 0.8, 1.0),
        max_crop_trials: int = 250,
        horizontal_flip_prob: float = 0.5,
    ) -> None:
        if image_size <= 0:
            raise ValueError("image_size must be positive")
        if max_crop_trials <= 0:
            raise ValueError("max_crop_trials must be positive")

        self.image_size = int(image_size)
        self.bgr_mean = _normalize_bgr_mean(bgr_mean)
        self.min_face_size = float(min_face_size)
        self.crop_scales = tuple(float(scale) for scale in crop_scales)
        self.max_crop_trials = int(max_crop_trials)
        self.horizontal_flip_prob = float(horizontal_flip_prob)

    def __call__(
        self, image: Tensor | Image.Image | np.ndarray, target: dict[str, Any]
    ) -> tuple[Tensor, dict[str, Any]]:
        image_np = _to_numpy_rgb_image(image)
        target_np = _target_to_numpy(target)

        image_np = _photometric_distort(image_np)
        image_np, target_np = _random_square_crop(
            image=image_np,
            target=target_np,
            image_size=self.image_size,
            min_face_size=self.min_face_size,
            crop_scales=self.crop_scales,
            max_trials=self.max_crop_trials,
        )
        image_np, target_np = _random_horizontal_flip(
            image=image_np,
            target=target_np,
            probability=self.horizontal_flip_prob,
        )
        image_np = _pad_to_square(image_np, fill=_bgr_mean_to_rgb_fill(self.bgr_mean))
        image_np, target_np = _resize_with_targets(
            image=image_np,
            target=target_np,
            output_size=self.image_size,
        )

        image_tensor = _to_bgr_mean_subtracted_tensor(image_np, self.bgr_mean)
        return image_tensor, _target_from_numpy(target, target_np)


class RetinaFaceEvalTransform:
    """RetinaFace evaluation/inference preprocessing without random augmentation."""

    def __init__(
        self,
        image_size: int = 640,
        bgr_mean: tuple[float, float, float] = (104.0, 117.0, 123.0),
        pad_to_square: bool = True,
    ) -> None:
        if image_size <= 0:
            raise ValueError("image_size must be positive")

        self.image_size = int(image_size)
        self.bgr_mean = _normalize_bgr_mean(bgr_mean)
        self.pad_to_square = bool(pad_to_square)

    def __call__(
        self, image: Tensor | Image.Image | np.ndarray, target: dict[str, Any]
    ) -> tuple[Tensor, dict[str, Any]]:
        image_np = _to_numpy_rgb_image(image)
        target_np = _target_to_numpy(target)

        if self.pad_to_square:
            image_np = _pad_to_square(
                image_np, fill=_bgr_mean_to_rgb_fill(self.bgr_mean)
            )

        image_np, target_np = _resize_with_targets(
            image=image_np,
            target=target_np,
            output_size=self.image_size,
        )

        image_tensor = _to_bgr_mean_subtracted_tensor(image_np, self.bgr_mean)
        return image_tensor, _target_from_numpy(target, target_np)


def _target_to_numpy(target: dict[str, Any]) -> dict[str, Any]:
    return {
        key: (
            _to_numpy_copy(value)
            if key
            in {"boxes", "labels", "landmarks", "landmark_valid", "annotation_score"}
            else value
        )
        for key, value in target.items()
    }


def _target_from_numpy(
    original: dict[str, Any], target_np: dict[str, Any]
) -> dict[str, Any]:
    converted = dict(original)
    for key, value in target_np.items():
        if key == "boxes":
            converted[key] = torch.from_numpy(value.astype(np.float32, copy=False))
        elif key == "labels":
            converted[key] = torch.from_numpy(value.astype(np.int64, copy=False))
        elif key == "landmarks":
            converted[key] = torch.from_numpy(value.astype(np.float32, copy=False))
        elif key == "landmark_valid":
            converted[key] = torch.from_numpy(value.astype(np.bool_, copy=False))
        elif key == "annotation_score":
            converted[key] = torch.from_numpy(value.astype(np.float32, copy=False))
        else:
            converted[key] = value
    return converted


def _to_numpy_copy(value: Any) -> np.ndarray:
    if isinstance(value, Tensor):
        return value.detach().cpu().numpy().copy()
    if isinstance(value, np.ndarray):
        return value.copy()
    raise TypeError(f"unsupported target value type: {type(value)!r}")


def _to_numpy_rgb_image(image: Tensor | Image.Image | np.ndarray) -> np.ndarray:
    if isinstance(image, Image.Image):
        array = np.asarray(image.convert("RGB"), dtype=np.uint8)
        return array.copy()

    if isinstance(image, Tensor):
        image = image.detach().cpu()
        if image.dim() != 3:
            raise ValueError("tensor image must have shape (C, H, W) or (H, W, C)")
        if image.shape[0] in {1, 3}:
            array = image.permute(1, 2, 0).numpy()
        else:
            array = image.numpy()
        if array.dtype != np.uint8:
            array = np.clip(array, 0, 255).astype(np.uint8)
        return array.copy()

    if isinstance(image, np.ndarray):
        if image.ndim != 3:
            raise ValueError("image array must have shape (H, W, C)")
        if image.dtype != np.uint8:
            image = np.clip(image, 0, 255).astype(np.uint8)
        return image.copy()

    raise TypeError(f"unsupported image type: {type(image)!r}")


def _photometric_distort(image: np.ndarray) -> np.ndarray:
    pil_image = Image.fromarray(image)

    if random.random() < 0.5:
        pil_image = ImageEnhance.Brightness(pil_image).enhance(
            random.uniform(0.75, 1.25)
        )

    contrast_first = random.random() < 0.5
    if contrast_first and random.random() < 0.5:
        pil_image = ImageEnhance.Contrast(pil_image).enhance(random.uniform(0.75, 1.25))

    if random.random() < 0.5:
        pil_image = ImageEnhance.Color(pil_image).enhance(random.uniform(0.75, 1.25))

    if random.random() < 0.5:
        hsv = np.asarray(pil_image.convert("HSV"), dtype=np.uint8).copy()
        hue_delta = random.randint(-18, 18)
        hsv[..., 0] = (hsv[..., 0].astype(np.int16) + hue_delta) % 255
        pil_image = Image.frombuffer(
            "HSV",
            pil_image.size,
            hsv.tobytes(),
            "raw",
            "HSV",
            0,
            1,
        ).convert("RGB")

    if not contrast_first and random.random() < 0.5:
        pil_image = ImageEnhance.Contrast(pil_image).enhance(random.uniform(0.75, 1.25))

    return np.asarray(pil_image, dtype=np.uint8)


def _random_square_crop(
    image: np.ndarray,
    target: dict[str, Any],
    image_size: int,
    min_face_size: float,
    crop_scales: tuple[float, ...],
    max_trials: int,
) -> tuple[np.ndarray, dict[str, Any]]:
    boxes = target["boxes"]
    if boxes.size == 0:
        return image, target

    height, width, _ = image.shape
    short_side = min(height, width)
    roi = np.zeros((1, 4), dtype=np.float32)

    for _ in range(max_trials):
        scale = random.choice(crop_scales)
        crop_size = int(scale * short_side)
        crop_size = max(1, min(crop_size, height, width))

        left = 0 if width == crop_size else random.randint(0, width - crop_size)
        top = 0 if height == crop_size else random.randint(0, height - crop_size)
        roi[0] = np.asarray(
            [left, top, left + crop_size, top + crop_size], dtype=np.float32
        )

        fully_inside = (
            matrix_iof(boxes.astype(np.float32, copy=False), roi)[:, 0] >= 1.0
        )
        if not fully_inside.any():
            continue

        boxes_t = boxes[fully_inside].copy()
        labels_t = target["labels"][fully_inside].copy()
        landmarks_t = target["landmarks"][fully_inside].copy()
        landmark_valid_t = target["landmark_valid"][fully_inside].copy()
        annotation_score_t = target["annotation_score"][fully_inside].copy()

        centers = (boxes_t[:, :2] + boxes_t[:, 2:]) / 2
        center_inside = (
            (centers[:, 0] > left)
            & (centers[:, 0] < left + crop_size)
            & (centers[:, 1] > top)
            & (centers[:, 1] < top + crop_size)
        )
        if not center_inside.any():
            continue

        boxes_t = boxes_t[center_inside]
        labels_t = labels_t[center_inside]
        landmarks_t = landmarks_t[center_inside]
        landmark_valid_t = landmark_valid_t[center_inside]
        annotation_score_t = annotation_score_t[center_inside]

        boxes_t[:, 0::2] -= left
        boxes_t[:, 1::2] -= top
        boxes_t[:, 0::2] = np.clip(boxes_t[:, 0::2], 0, crop_size)
        boxes_t[:, 1::2] = np.clip(boxes_t[:, 1::2], 0, crop_size)

        landmarks_t = _translate_landmarks(
            landmarks_t,
            landmark_valid_t,
            dx=-left,
            dy=-top,
        )

        resized_face_sizes = (
            np.minimum(
                boxes_t[:, 2] - boxes_t[:, 0],
                boxes_t[:, 3] - boxes_t[:, 1],
            )
            * float(image_size)
            / float(crop_size)
        )
        keep = resized_face_sizes >= min_face_size
        if not keep.any():
            continue

        cropped_image = image[top : top + crop_size, left : left + crop_size]
        cropped_target = dict(target)
        cropped_target["boxes"] = boxes_t[keep]
        cropped_target["labels"] = labels_t[keep]
        cropped_target["landmarks"] = landmarks_t[keep]
        cropped_target["landmark_valid"] = landmark_valid_t[keep]
        cropped_target["annotation_score"] = annotation_score_t[keep]
        return cropped_image, cropped_target

    return image, target


def _random_horizontal_flip(
    image: np.ndarray,
    target: dict[str, Any],
    probability: float,
) -> tuple[np.ndarray, dict[str, Any]]:
    if random.random() >= probability:
        return image, target

    flipped_image = np.ascontiguousarray(image[:, ::-1])
    height, width, _ = flipped_image.shape

    boxes = target["boxes"].copy()
    if boxes.size > 0:
        old_x1 = boxes[:, 0].copy()
        old_x2 = boxes[:, 2].copy()
        boxes[:, 0] = width - old_x2
        boxes[:, 2] = width - old_x1

    landmarks = target["landmarks"].copy().reshape(-1, 5, 2)
    landmark_valid = target["landmark_valid"].copy()
    if landmarks.size > 0:
        valid_mask = landmark_valid
        landmarks[:, :, 0] = np.where(valid_mask, width - landmarks[:, :, 0], -1.0)
        reorder = np.asarray([1, 0, 2, 4, 3], dtype=np.int64)
        landmarks = landmarks[:, reorder, :]
        landmark_valid = landmark_valid[:, reorder]
        landmarks = np.where(landmark_valid[:, :, None], landmarks, -1.0)

    flipped_target = dict(target)
    flipped_target["boxes"] = boxes
    flipped_target["landmarks"] = landmarks.reshape(-1, 10)
    flipped_target["landmark_valid"] = landmark_valid
    return flipped_image, flipped_target


def _pad_to_square(
    image: np.ndarray,
    fill: tuple[float, float, float],
) -> np.ndarray:
    height, width, channels = image.shape
    if height == width:
        return image

    side = max(height, width)
    canvas = np.empty((side, side, channels), dtype=np.uint8)
    canvas[...] = np.asarray(fill, dtype=np.uint8)
    canvas[:height, :width] = image
    return canvas


def _bgr_mean_to_rgb_fill(
    bgr_mean: tuple[float, float, float],
) -> tuple[float, float, float]:
    return float(bgr_mean[2]), float(bgr_mean[1]), float(bgr_mean[0])


def _resize_with_targets(
    image: np.ndarray,
    target: dict[str, Any],
    output_size: int,
) -> tuple[np.ndarray, dict[str, Any]]:
    original_height, original_width = image.shape[:2]
    resized_image = np.asarray(
        Image.fromarray(image).resize(
            (output_size, output_size), resample=Image.Resampling.BILINEAR
        ),
        dtype=np.uint8,
    )

    scale_x = float(output_size) / float(original_width)
    scale_y = float(output_size) / float(original_height)

    resized_target = dict(target)
    boxes = target["boxes"].copy()
    if boxes.size > 0:
        boxes[:, 0::2] *= scale_x
        boxes[:, 1::2] *= scale_y
    resized_target["boxes"] = boxes

    landmarks = target["landmarks"].copy()
    landmark_valid = target["landmark_valid"].copy()
    if landmarks.size > 0:
        landmarks = _scale_landmarks(
            landmarks,
            landmark_valid,
            scale_x=scale_x,
            scale_y=scale_y,
        )
    resized_target["landmarks"] = landmarks
    resized_target["landmark_valid"] = landmark_valid
    return resized_image, resized_target


def _translate_landmarks(
    landmarks: np.ndarray,
    landmark_valid: np.ndarray,
    dx: float,
    dy: float,
) -> np.ndarray:
    translated = landmarks.copy().reshape(-1, 5, 2)
    if translated.size == 0:
        return landmarks.copy()

    valid_mask = landmark_valid
    translated[:, :, 0] = np.where(valid_mask, translated[:, :, 0] + dx, -1.0)
    translated[:, :, 1] = np.where(valid_mask, translated[:, :, 1] + dy, -1.0)
    return translated.reshape(-1, 10)


def _scale_landmarks(
    landmarks: np.ndarray,
    landmark_valid: np.ndarray,
    scale_x: float,
    scale_y: float,
) -> np.ndarray:
    scaled = landmarks.copy().reshape(-1, 5, 2)
    if scaled.size == 0:
        return landmarks.copy()

    valid_mask = landmark_valid
    scaled[:, :, 0] = np.where(valid_mask, scaled[:, :, 0] * scale_x, -1.0)
    scaled[:, :, 1] = np.where(valid_mask, scaled[:, :, 1] * scale_y, -1.0)
    return scaled.reshape(-1, 10)


def _to_bgr_mean_subtracted_tensor(
    image: np.ndarray,
    bgr_mean: tuple[float, float, float],
) -> Tensor:
    image = image.astype(np.float32, copy=False)
    image = image[:, :, ::-1]  # RGB -> BGR
    image -= np.asarray(bgr_mean, dtype=np.float32)
    return torch.from_numpy(np.ascontiguousarray(image)).permute(2, 0, 1)

# 5. WiderFace Dataset

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable

import numpy as np
import torch
from PIL import Image
from torch import Tensor
from torch.utils.data import Dataset

__all__ = ["WiderFaceDataset", "widerface_collate"]


@dataclass(frozen=True)
class _WiderFaceRecord:
    relative_path: str
    boxes: np.ndarray
    landmarks: np.ndarray
    landmark_valid: np.ndarray
    annotation_score: np.ndarray


class WiderFaceDataset(Dataset[tuple[Tensor | Image.Image, dict[str, Any]]]):
    """
    WIDER FACE training dataset parser for RetinaFace-style annotations.

    The train annotation file is expected to look like:
    - ``# relative/image/path.jpg``
    - one or more rows of ``xywh + 5 * (x, y, flag) + score``

    Returned targets are compatible with ``MultiBoxLoss``:
    - ``boxes``: ``(N, 4)``
    - ``labels``: ``(N,)`` with foreground label ``1``
    - ``landmarks``: ``(N, 10)``
    - ``landmark_valid``: ``(N, 5)``

    If ``normalize_targets=True``, boxes and valid landmark coordinates are
    normalized after ``transform`` using the current image size.
    """

    def __init__(
        self,
        annotation_file: str | Path = "datasets/train/train_annotations.txt",
        image_root: str | Path | None = None,
        transform: (
            Callable[
                [Tensor | Image.Image, dict[str, Any]],
                tuple[Tensor | Image.Image, dict[str, Any]],
            ]
            | None
        ) = None,
        normalize_targets: bool = True,
        to_tensor: bool = True,
    ) -> None:
        self.annotation_file = Path(annotation_file)
        if not self.annotation_file.is_file():
            raise FileNotFoundError(
                f"annotation file not found: {self.annotation_file}"
            )

        self.image_root = (
            Path(image_root)
            if image_root is not None
            else self.annotation_file.parent / "images"
        )
        if not self.image_root.is_dir():
            raise FileNotFoundError(f"image root not found: {self.image_root}")

        self.transform = transform
        self.normalize_targets = normalize_targets
        self.to_tensor = to_tensor
        self.records = self._parse_train_annotations(self.annotation_file)

    @classmethod
    def from_train_split(
        cls,
        train_root: str | Path = "datasets/train",
        **kwargs: Any,
    ) -> "WiderFaceDataset":
        train_root = Path(train_root)
        return cls(
            annotation_file=train_root / "train_annotations.txt",
            image_root=train_root / "images",
            **kwargs,
        )

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[Tensor | Image.Image, dict[str, Any]]:
        record = self.records[index]
        image_path = self.image_root / record.relative_path.lstrip("/").lstrip("./")
        if not image_path.is_file():
            raise FileNotFoundError(f"image not found: {image_path}")

        image = Image.open(image_path).convert("RGB")
        orig_width, orig_height = image.size

        target = {
            "boxes": torch.from_numpy(record.boxes.copy()),
            "labels": torch.ones((record.boxes.shape[0],), dtype=torch.long),
            "landmarks": torch.from_numpy(record.landmarks.copy()),
            "landmark_valid": torch.from_numpy(record.landmark_valid.copy()),
            "annotation_score": torch.from_numpy(record.annotation_score.copy()),
            "image_id": torch.tensor(index, dtype=torch.long),
            "orig_size": torch.tensor([orig_height, orig_width], dtype=torch.long),
            "image_size": torch.tensor([orig_height, orig_width], dtype=torch.long),
            "path": record.relative_path,
        }

        if self.transform is not None:
            image, target = self.transform(image, target)

        current_height, current_width = self._get_image_size(image)
        target["image_size"] = torch.tensor(
            [current_height, current_width], dtype=torch.long
        )

        if self.normalize_targets:
            self._normalize_target_in_place(target, current_height, current_width)

        if self.to_tensor:
            image = self._image_to_tensor(image)

        return image, target

    @staticmethod
    def _parse_train_annotations(annotation_file: Path) -> list[_WiderFaceRecord]:
        records: list[_WiderFaceRecord] = []
        current_path: str | None = None
        current_rows: list[np.ndarray] = []

        with annotation_file.open("r", encoding="utf-8") as f:
            for line_no, raw_line in enumerate(f, start=1):
                line = raw_line.strip()
                if not line:
                    continue

                if line.startswith("#"):
                    if current_path is not None:
                        records.append(
                            WiderFaceDataset._build_record(current_path, current_rows)
                        )
                    current_path = line[1:].strip()
                    current_rows = []
                    continue

                if current_path is None:
                    raise ValueError(
                        f"invalid annotation format at line {line_no}: "
                        "expected image path starting with '#'"
                    )

                fields = line.split()
                if len(fields) < 20:
                    raise ValueError(
                        f"invalid annotation row at line {line_no}: "
                        f"expected at least 20 values, got {len(fields)}"
                    )
                current_rows.append(np.asarray(fields[:20], dtype=np.float32))

        if current_path is not None:
            records.append(WiderFaceDataset._build_record(current_path, current_rows))

        if not records:
            raise ValueError(f"no records found in {annotation_file}")

        return records

    @staticmethod
    def _build_record(relative_path: str, rows: list[np.ndarray]) -> _WiderFaceRecord:
        if not rows:
            return _WiderFaceRecord(
                relative_path=relative_path,
                boxes=np.zeros((0, 4), dtype=np.float32),
                landmarks=np.zeros((0, 10), dtype=np.float32),
                landmark_valid=np.zeros((0, 5), dtype=np.bool_),
                annotation_score=np.zeros((0,), dtype=np.float32),
            )

        boxes: list[np.ndarray] = []
        landmarks: list[np.ndarray] = []
        landmark_valid: list[np.ndarray] = []
        annotation_score: list[float] = []

        for row in rows:
            x, y, w, h = row[:4]
            if w <= 0 or h <= 0:
                continue

            boxes.append(np.asarray([x, y, x + w, y + h], dtype=np.float32))

            landmark_triplets = row[4:19].reshape(5, 3)
            landmark_xy = landmark_triplets[:, :2].reshape(10).astype(np.float32)
            valid_points = (
                (landmark_triplets[:, 2] >= 0)
                & (landmark_triplets[:, 0] >= 0)
                & (landmark_triplets[:, 1] >= 0)
            )

            landmarks.append(landmark_xy)
            landmark_valid.append(valid_points.astype(np.bool_))
            annotation_score.append(float(row[19]))

        if not boxes:
            return _WiderFaceRecord(
                relative_path=relative_path,
                boxes=np.zeros((0, 4), dtype=np.float32),
                landmarks=np.zeros((0, 10), dtype=np.float32),
                landmark_valid=np.zeros((0, 5), dtype=np.bool_),
                annotation_score=np.zeros((0,), dtype=np.float32),
            )

        return _WiderFaceRecord(
            relative_path=relative_path,
            boxes=np.stack(boxes).astype(np.float32, copy=False),
            landmarks=np.stack(landmarks).astype(np.float32, copy=False),
            landmark_valid=np.stack(landmark_valid).astype(np.bool_, copy=False),
            annotation_score=np.asarray(annotation_score, dtype=np.float32),
        )

    @staticmethod
    def _normalize_target_in_place(
        target: dict[str, Any], height: int, width: int
    ) -> None:
        if height <= 0 or width <= 0:
            raise ValueError("image height and width must be positive")

        boxes = torch.as_tensor(target["boxes"], dtype=torch.float32)
        landmarks = torch.as_tensor(target["landmarks"], dtype=torch.float32)
        landmark_valid = torch.as_tensor(target["landmark_valid"], dtype=torch.bool)

        if boxes.numel() > 0:
            boxes = boxes.clone()
            boxes[:, 0::2] /= float(width)
            boxes[:, 1::2] /= float(height)

        if landmarks.numel() > 0:
            landmarks = landmarks.clone()
            coord_valid = landmark_valid.unsqueeze(-1).expand(-1, -1, 2).reshape(-1, 10)
            scale = landmarks.new_tensor([width, height] * 5)
            landmarks[coord_valid] = (
                landmarks[coord_valid] / scale.expand_as(landmarks)[coord_valid]
            )

        target["boxes"] = boxes
        target["landmarks"] = landmarks
        target["landmark_valid"] = landmark_valid

    @staticmethod
    def _get_image_size(image: Tensor | Image.Image | np.ndarray) -> tuple[int, int]:
        if isinstance(image, Image.Image):
            width, height = image.size
            return height, width

        if isinstance(image, Tensor):
            if image.dim() == 2:
                return int(image.shape[0]), int(image.shape[1])
            if image.dim() == 3:
                if image.shape[0] <= 4:
                    return int(image.shape[1]), int(image.shape[2])
                return int(image.shape[0]), int(image.shape[1])
            raise ValueError("tensor image must have shape (H, W) or (C, H, W)")

        if isinstance(image, np.ndarray):
            if image.ndim == 2:
                return int(image.shape[0]), int(image.shape[1])
            if image.ndim == 3:
                if image.shape[0] <= 4:
                    return int(image.shape[1]), int(image.shape[2])
                return int(image.shape[0]), int(image.shape[1])
            raise ValueError("ndarray image must have shape (H, W) or (H, W, C)")

        raise TypeError(f"unsupported image type: {type(image)!r}")

    @staticmethod
    def _image_to_tensor(image: Tensor | Image.Image | np.ndarray) -> Tensor:
        if isinstance(image, Tensor):
            if image.is_floating_point():
                return image
            return image.float() / 255.0

        if isinstance(image, Image.Image):
            array = np.asarray(image, dtype=np.float32)
        elif isinstance(image, np.ndarray):
            array = image.astype(np.float32, copy=False)
        else:
            raise TypeError(f"unsupported image type: {type(image)!r}")

        if array.ndim == 2:
            tensor = torch.from_numpy(array).unsqueeze(0)
        elif array.ndim == 3:
            tensor = torch.from_numpy(array).permute(2, 0, 1)
        else:
            raise ValueError("image array must have shape (H, W) or (H, W, C)")

        return tensor.float() / 255.0


def widerface_collate(
    batch: list[tuple[Tensor | Image.Image, dict[str, Any]]],
) -> tuple[Tensor | list[Tensor | Image.Image], list[dict[str, Any]]]:
    images, targets = zip(*batch)
    if all(isinstance(image, Tensor) for image in images):
        image_shapes = {tuple(image.shape) for image in images}
        if len(image_shapes) == 1:
            return torch.stack(list(images), dim=0), list(targets)
    return list(images), list(targets)

# 6. Multibox Loss

In [ ]:
from __future__ import annotations

from collections.abc import Mapping, Sequence

import torch
import torch.nn.functional as F
from torch import Tensor, nn


__all__ = ["MultiBoxLoss"]


class MultiBoxLoss(nn.Module):
    """
    RetinaFace/SSD-style multibox loss with landmark visibility masking.

    Expected predictions:
    - ``loc_data``: ``(batch, num_priors, 4)``
    - ``conf_data``: ``(batch, num_priors, num_classes)``
    - ``landm_data``: ``(batch, num_priors, 10)``

    Supported target formats per image:
    - ``Tensor[N, 15]``: ``[x1, y1, x2, y2, lm10, label]``
    - ``Tensor[N, 20]``: ``[x1, y1, x2, y2, lm10, label, valid5]``
    - ``Tensor[N, 25]``: ``[x1, y1, x2, y2, lm10, label, valid10]``
    - ``dict`` with keys:
      ``boxes``, ``labels`` (optional), ``landms``/``landmarks`` (optional),
      ``landm_valid``/``landmark_valid`` (optional)

    Boxes must already be in point-form and normalized to the same scale as priors.
    """

    def __init__(
        self,
        cfg: Mapping[str, object],
        num_classes: int = 2,
        overlap_thresh: float = 0.30,
        neg_pos_ratio: int = 7,
        best_prior_iou_floor: float = 0.1,
        landm_weight: float = 1.0,
    ) -> None:
        super().__init__()

        if num_classes < 2:
            raise ValueError("num_classes must be at least 2")
        if neg_pos_ratio <= 0:
            raise ValueError("neg_pos_ratio must be positive")

        variance = cfg.get("variance")
        if not isinstance(variance, Sequence) or len(variance) != 2:
            raise ValueError("cfg['variance'] must be a sequence of length 2")

        self.num_classes = num_classes
        self.threshold = overlap_thresh
        self.neg_pos_ratio = neg_pos_ratio
        self.best_prior_iou_floor = best_prior_iou_floor
        self.variance = (float(variance[0]), float(variance[1]))
        self.loc_weight = float(cfg.get("loc_weight", 1.0))
        self.landm_weight = float(landm_weight)

    def forward(
        self,
        predictions: tuple[Tensor, Tensor, Tensor],
        priors: Tensor,
        targets: Sequence[Tensor | Mapping[str, Tensor]],
    ) -> tuple[Tensor, Tensor, Tensor]:
        loc_data, conf_data, landm_data = predictions

        if loc_data.dim() != 3 or loc_data.size(-1) != 4:
            raise ValueError("loc_data must have shape (batch, num_priors, 4)")
        if conf_data.dim() != 3 or conf_data.size(-1) != self.num_classes:
            raise ValueError(
                "conf_data must have shape (batch, num_priors, num_classes)"
            )
        if landm_data.dim() != 3 or landm_data.size(-1) != 10:
            raise ValueError("landm_data must have shape (batch, num_priors, 10)")

        if priors.dim() == 3:
            if priors.size(0) != 1:
                raise ValueError(
                    "priors must have shape (num_priors, 4) or (1, num_priors, 4)"
                )
            priors = priors.squeeze(0)
        if priors.dim() != 2 or priors.size(-1) != 4:
            raise ValueError("priors must have shape (num_priors, 4)")

        batch_size, num_priors, _ = loc_data.shape
        if len(targets) != batch_size:
            raise ValueError("targets length must match batch size")
        if conf_data.size(1) != num_priors or landm_data.size(1) != num_priors:
            raise ValueError("prediction heads must agree on num_priors")
        if priors.size(0) != num_priors:
            raise ValueError("priors count must match prediction num_priors")

        device = loc_data.device
        priors = priors.to(device=device, dtype=loc_data.dtype).detach()

        with torch.no_grad():
            loc_t = loc_data.new_zeros((batch_size, num_priors, 4))
            conf_t = torch.zeros(
                (batch_size, num_priors), dtype=torch.long, device=device
            )
            landm_t = landm_data.new_zeros((batch_size, num_priors, 10))
            landm_valid_t = torch.zeros(
                (batch_size, num_priors, 10), dtype=torch.bool, device=device
            )

            for batch_idx, target in enumerate(targets):
                truths, labels, landms, landm_valid = self._unpack_target(
                    target=target,
                    device=device,
                    box_dtype=loc_data.dtype,
                )
                landms_for_match = landms.clone()
                landms_for_match[~landm_valid] = -1

                match(
                    threshold=self.threshold,
                    truths=truths,
                    priors=priors,
                    variances=self.variance,
                    labels=labels,
                    landms=landms_for_match,
                    loc_t=loc_t,
                    conf_t=conf_t,
                    landm_t=landm_t,
                    idx=batch_idx,
                    landm_valid_t=landm_valid_t,
                    best_prior_iou_floor=self.best_prior_iou_floor,
                )

        positive_mask = conf_t > 0
        num_pos_per_image = positive_mask.sum(dim=1, keepdim=True)
        num_pos_total = num_pos_per_image.sum().clamp(min=1).to(loc_data.dtype)

        loss_loc = self._localization_loss(
            loc_data=loc_data,
            loc_t=loc_t,
            positive_mask=positive_mask,
            normalizer=num_pos_total,
        )
        loss_conf = self._classification_loss(
            conf_data=conf_data,
            conf_t=conf_t,
            positive_mask=positive_mask,
            num_pos_per_image=num_pos_per_image,
            normalizer=num_pos_total,
        )
        loss_landm = self._landmark_loss(
            landm_data=landm_data,
            landm_t=landm_t,
            landm_valid_t=landm_valid_t,
        )

        return loss_loc, loss_conf, loss_landm

    def combine_losses(
        self, loss_loc: Tensor, loss_conf: Tensor, loss_landm: Tensor
    ) -> Tensor:
        """Combine normalized losses using the configured weights."""
        return self.loc_weight * loss_loc + loss_conf + self.landm_weight * loss_landm

    def _localization_loss(
        self,
        loc_data: Tensor,
        loc_t: Tensor,
        positive_mask: Tensor,
        normalizer: Tensor,
    ) -> Tensor:
        if not positive_mask.any():
            return loc_data.sum() * 0

        loss_loc = F.smooth_l1_loss(
            loc_data[positive_mask],
            loc_t[positive_mask],
            reduction="sum",
        )
        return loss_loc / normalizer

    def _classification_loss(
        self,
        conf_data: Tensor,
        conf_t: Tensor,
        positive_mask: Tensor,
        num_pos_per_image: Tensor,
        normalizer: Tensor,
    ) -> Tensor:
        batch_size, num_priors, _ = conf_data.shape

        with torch.no_grad():
            mining_logits = conf_data.detach().view(-1, self.num_classes)
            mining_loss = log_sum_exp(mining_logits)
            mining_loss -= mining_logits.gather(1, conf_t.view(-1, 1))
            mining_loss = mining_loss.view(batch_size, num_priors)
            mining_loss[positive_mask] = -torch.inf

            _, loss_idx = mining_loss.sort(dim=1, descending=True)
            _, idx_rank = loss_idx.sort(dim=1)

            num_neg = torch.clamp(
                self.neg_pos_ratio * num_pos_per_image,
                max=max(num_priors - 1, 0),
            )
            negative_mask = idx_rank < num_neg.expand_as(idx_rank)

        sample_mask = positive_mask | negative_mask
        if not sample_mask.any():
            return conf_data.sum() * 0

        loss_conf = F.cross_entropy(
            conf_data[sample_mask].view(-1, self.num_classes),
            conf_t[sample_mask],
            reduction="sum",
        )
        return loss_conf / normalizer

    def _landmark_loss(
        self,
        landm_data: Tensor,
        landm_t: Tensor,
        landm_valid_t: Tensor,
    ) -> Tensor:
        if not landm_valid_t.any():
            return landm_data.sum() * 0

        loss_landm = F.smooth_l1_loss(
            landm_data[landm_valid_t],
            landm_t[landm_valid_t],
            reduction="sum",
        )

        # Ten valid coordinates correspond to one fully supervised face.
        valid_face_equivalents = (
            landm_valid_t.sum().to(landm_data.dtype) / 10.0
        ).clamp(min=1.0)
        return loss_landm / valid_face_equivalents

    def _unpack_target(
        self,
        target: Tensor | Mapping[str, Tensor],
        device: torch.device,
        box_dtype: torch.dtype,
    ) -> tuple[Tensor, Tensor, Tensor, Tensor]:
        if isinstance(target, Tensor):
            truths, labels, landms, landm_valid = self._unpack_tensor_target(target)
        elif isinstance(target, Mapping):
            truths, labels, landms, landm_valid = self._unpack_mapping_target(target)
        else:
            raise TypeError("each target must be a Tensor or mapping of tensors")

        truths = truths.to(device=device, dtype=box_dtype).reshape(-1, 4)
        labels = labels.to(device=device, dtype=torch.long).view(-1)
        landms = landms.to(device=device, dtype=box_dtype).reshape(-1, 10)
        if truths.size(0) != labels.size(0) or truths.size(0) != landms.size(0):
            raise ValueError(
                "boxes, labels, and landmarks must have the same number of rows"
            )
        landm_valid = self._expand_landmark_validity(
            landm_valid=landm_valid,
            landms=landms,
            device=device,
        )

        if truths.numel() == 0:
            return (
                truths.reshape(0, 4),
                labels.reshape(0),
                landms.reshape(0, 10),
                landm_valid.reshape(0, 10),
            )

        keep = (
            (labels > 0) & (truths[:, 2] > truths[:, 0]) & (truths[:, 3] > truths[:, 1])
        )
        truths = truths[keep]
        labels = labels[keep]
        landms = landms[keep]
        landm_valid = landm_valid[keep]

        if labels.numel() > 0 and labels.max().item() >= self.num_classes:
            raise ValueError("target labels must be in [0, num_classes - 1]")

        return truths, labels, landms, landm_valid

    def _unpack_tensor_target(
        self, target: Tensor
    ) -> tuple[Tensor, Tensor, Tensor, Tensor | None]:
        if target.dim() != 2:
            raise ValueError("tensor targets must have shape (num_gt, num_fields)")
        if target.numel() == 0:
            empty = target.new_zeros((0,))
            return (
                target.new_zeros((0, 4)),
                empty.to(dtype=torch.long),
                target.new_zeros((0, 10)),
                target.new_zeros((0, 10), dtype=torch.bool),
            )

        num_fields = target.size(1)
        if num_fields not in {15, 20, 25}:
            raise ValueError(
                "tensor targets must have 15, 20, or 25 columns: "
                "[box4, landm10, label, optional validity]"
            )

        truths = target[:, :4]
        landms = target[:, 4:14]
        labels = target[:, 14]
        landm_valid: Tensor | None = None
        if num_fields == 20:
            landm_valid = target[:, 15:20]
        elif num_fields == 25:
            landm_valid = target[:, 15:25]

        return truths, labels, landms, landm_valid

    def _unpack_mapping_target(
        self, target: Mapping[str, Tensor]
    ) -> tuple[Tensor, Tensor, Tensor, Tensor | None]:
        truths = target.get("boxes")
        if truths is None:
            truths = target.get("truths")
        if truths is None:
            raise KeyError("target mapping must contain 'boxes' or 'truths'")

        landms = target.get("landms")
        if landms is None:
            landms = target.get("landmarks")

        labels = target.get("labels")
        if labels is None:
            labels = torch.ones(
                (truths.size(0),), device=truths.device, dtype=truths.dtype
            )

        landm_valid = target.get("landm_valid")
        if landm_valid is None:
            landm_valid = target.get("landmark_valid")
        if landm_valid is None:
            landm_valid = target.get("landmarks_valid")

        if landms is None:
            landms = torch.full(
                (truths.size(0), 10), -1, device=truths.device, dtype=truths.dtype
            )

        return truths, labels, landms, landm_valid

    def _expand_landmark_validity(
        self,
        landm_valid: Tensor | None,
        landms: Tensor,
        device: torch.device,
    ) -> Tensor:
        if landm_valid is None:
            point_valid = landms.view(-1, 5, 2).ge(0).all(dim=2)
            return point_valid.unsqueeze(-1).expand(-1, -1, 2).reshape(-1, 10)

        landm_valid = landm_valid.to(device=device)
        if landm_valid.dim() != 2:
            raise ValueError(
                "landmark validity must have shape (num_gt, 5) or (num_gt, 10)"
            )
        if landm_valid.size(0) != landms.size(0):
            raise ValueError("landmark validity rows must match number of targets")
        if landm_valid.size(1) == 5:
            point_valid = landm_valid > 0
            return point_valid.unsqueeze(-1).expand(-1, -1, 2).reshape(-1, 10)
        if landm_valid.size(1) == 10:
            return landm_valid > 0
        raise ValueError("landmark validity must have 5 or 10 columns")

# 7. Model Retinaface

## Utils

In [ ]:
from collections import OrderedDict

from torch import nn, Tensor


def _make_divisible(v: float, divisor: int = 8) -> int:
    """This function ensures that all layers have a channel number divisible by 8"""
    new_v = max(divisor, int(v + divisor / 2) // divisor * divisor)
    # Make sure that round down does not go down by more than 10%.
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


class IntermediateLayerGetterByIndex(nn.Module):
    def __init__(self, model: nn.Module, indexes: list[int] | None = None) -> None:
        super().__init__()
        features = getattr(model, "features", None)
        if not isinstance(features, nn.Sequential):
            raise TypeError("model.features must be an nn.Sequential")
        self.features: nn.Sequential = features
        self.indexes = tuple(indexes or [6, 13, 18])

    def forward(self, x: Tensor) -> OrderedDict[str, Tensor]:
        outputs: OrderedDict[str, Tensor] = OrderedDict()
        for idx, layer in enumerate(self.features):
            x = layer(x)
            if idx in self.indexes:
                out_name = f"layer_{idx}"
                outputs[out_name] = x

        return outputs

## Layers

In [ ]:
import torch
from torch import nn
from typing import Any, Callable


class Conv2dNormActivation(nn.Sequential):
    """Convolutional block, consists of nn.Conv2d, nn.BatchNorm2d, nn.ReLU"""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int | None = None,
        groups: int = 1,
        norm_layer: Callable[..., torch.nn.Module] | None = torch.nn.BatchNorm2d,
        activation_layer: Callable[..., torch.nn.Module] | None = torch.nn.LeakyReLU,
        dilation: int = 1,
        inplace: bool | None = True,
        negative_slope: float | None = None,
        bias: bool = False,
    ) -> None:
        if padding is None:
            padding = (kernel_size - 1) // 2 * dilation

        layers: list[nn.Module] = [
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                dilation=dilation,
                groups=groups,
                bias=bias,
            )
        ]
        if norm_layer is not None:
            layers.append(norm_layer(out_channels))

        if activation_layer is not None:
            params: dict[str, Any] = {} if inplace is None else {"inplace": inplace}
            if negative_slope is not None:
                params["negative_slope"] = negative_slope
            layers.append(activation_layer(**params))
        super().__init__(*layers)
        self.in_channels = in_channels
        self.out_channels = out_channels

## Backbone

In [ ]:
from typing import Any

import torch
from torch import nn, Tensor
from torchvision.models import MobileNet_V2_Weights

__all__ = ["mobilenet_v2"]


class InvertedResidual(nn.Module):
    def __init__(
        self, in_planes: int, out_planes: int, stride: int, expand_ratio: int
    ) -> None:
        super().__init__()
        self.stride = stride
        if stride not in [1, 2]:
            raise ValueError(f"stride should be 1 or 2 instead of {stride}")

        hidden_dim = int(round(in_planes * expand_ratio))
        self.use_res_connect = self.stride == 1 and in_planes == out_planes

        layers: list[nn.Module] = []
        if expand_ratio != 1:
            # pw
            layers.append(
                Conv2dNormActivation(
                    in_planes, hidden_dim, kernel_size=1, activation_layer=nn.ReLU6
                )
            )
        layers.extend(
            [
                # dw
                Conv2dNormActivation(
                    hidden_dim,
                    hidden_dim,
                    stride=stride,
                    groups=hidden_dim,
                    activation_layer=nn.ReLU6,
                ),
                # pw-linear
                nn.Conv2d(hidden_dim, out_planes, 1, 1, 0, bias=False),
                nn.BatchNorm2d(out_planes),
            ]
        )
        self.conv = nn.Sequential(*layers)
        self.out_channels = out_planes
        self._is_cn = stride > 1

    def forward(self, x: Tensor) -> Tensor:
        if self.use_res_connect:
            return x + self.conv(x)
        else:
            return self.conv(x)


class MobileNetV2(nn.Module):
    def __init__(
        self,
        num_classes: int = 1000,
        width_mult: float = 1.0,
        inverted_residual_setting: list[list[int]] | None = None,
        round_nearest: int = 8,
        dropout: float = 0.2,
    ) -> None:
        """
        MobileNet V2 main class

        Args:
            num_classes (int): Number of classes
            width_mult (float): Width multiplier - adjusts number of channels in each layer by this amount
            inverted_residual_setting: Network structure
            round_nearest (int): Round the number of channels in each layer to be a multiple of this number
            Set to 1 to turn off rounding
            block: Module specifying inverted residual building block for mobilenet
            dropout (float): The droupout probability

        """
        super().__init__()

        input_channel = 32
        last_channel = 1280

        if inverted_residual_setting is None:
            inverted_residual_setting = [
                # t, c, n, s
                [1, 16, 1, 1],
                [6, 24, 2, 2],
                [6, 32, 3, 2],
                [6, 64, 4, 2],
                [6, 96, 3, 1],
                [6, 160, 3, 2],
                [6, 320, 1, 1],
            ]

        # only check the first element, assuming user knows t,c,n,s are required
        if (
            len(inverted_residual_setting) == 0
            or len(inverted_residual_setting[0]) != 4
        ):
            raise ValueError(
                f"inverted_residual_setting should be non-empty or a 4-element list, got {inverted_residual_setting}"
            )

        # building first layer
        input_channel = _make_divisible(input_channel * width_mult, round_nearest)
        self.last_channel = _make_divisible(
            last_channel * max(1.0, width_mult), round_nearest
        )
        features: list[nn.Module] = [
            Conv2dNormActivation(3, input_channel, stride=2, activation_layer=nn.ReLU6)
        ]
        # building inverted residual blocks
        for t, c, n, s in inverted_residual_setting:
            output_channel = _make_divisible(c * width_mult, round_nearest)
            for i in range(n):
                stride = s if i == 0 else 1
                features.append(
                    InvertedResidual(
                        input_channel, output_channel, stride, expand_ratio=t
                    )
                )
                input_channel = output_channel
        # building last several layers
        features.append(
            Conv2dNormActivation(
                input_channel,
                self.last_channel,
                kernel_size=1,
                activation_layer=nn.ReLU6,
            )
        )
        # make it nn.Sequential
        self.features = nn.Sequential(*features)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # building classifier
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.last_channel, num_classes),
        )

        # weight initialization
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        x = self.classifier(x)
        return x


def mobilenet_v2(
    *, pretrained: bool = True, progress: bool = True, **kwargs: Any
) -> MobileNetV2:

    if pretrained:
        weights = MobileNet_V2_Weights.IMAGENET1K_V2
    else:
        weights = None

    model = MobileNetV2(**kwargs)

    if weights is not None:
        state_dict = weights.get_state_dict(progress=progress, check_hash=True)
        model.load_state_dict(state_dict)

    return model


# if __name__ == "__main__":
#     model = mobilenet_v2()
#     print(model)
#     x = torch.randn(1, 3, 640, 640)
#     t = IntermediateLayerGetterByIndex(model, [6, 13, 18])

#     a, b, c = list(t(x).values())

#     print(a.size())
#     print(b.size())
#     print(c.size())

## Heads

In [ ]:
import torch
from torch import Tensor, nn


class BboxHead(nn.Module):
    def __init__(
        self, in_channels: int = 512, num_anchors: int = 2, fpn_num: int = 3
    ) -> None:
        super().__init__()
        self.bbox_head = nn.ModuleList(
            [
                nn.Conv2d(
                    in_channels=in_channels,
                    out_channels=num_anchors * 4,
                    kernel_size=(1, 1),
                    stride=1,
                    padding=0,
                )
                for _ in range(fpn_num)
            ]
        )

    def forward(self, x: list[Tensor]) -> Tensor:
        output_tensors: list[Tensor] = []
        for feature, layer in zip(x, self.bbox_head):
            output_tensors.append(layer(feature).permute(0, 2, 3, 1).contiguous())

        bbox_offsets = torch.cat(
            [out.view(out.shape[0], -1, 4) for out in output_tensors], dim=1
        )
        return bbox_offsets

In [ ]:
import torch
from torch import Tensor, nn


class ClassHead(nn.Module):
    def __init__(
        self, in_channels: int = 512, num_anchors: int = 2, fpn_num: int = 3
    ) -> None:
        super().__init__()
        self.class_head = nn.ModuleList(
            [
                nn.Conv2d(
                    in_channels=in_channels,
                    out_channels=num_anchors * 2,
                    kernel_size=(1, 1),
                    stride=1,
                    padding=0,
                )
                for _ in range(fpn_num)
            ]
        )

    def forward(self, x: list[Tensor]) -> Tensor:
        output_tensors: list[Tensor] = []
        for feature, layer in zip(x, self.class_head):
            output_tensors.append(layer(feature).permute(0, 2, 3, 1).contiguous())

        class_scores = torch.cat(
            [out.view(out.shape[0], -1, 2) for out in output_tensors], dim=1
        )
        return class_scores

In [ ]:
import torch
from torch import Tensor, nn


class LandmarkHead(nn.Module):
    def __init__(
        self, in_channels: int = 512, num_anchors: int = 2, fpn_num: int = 3
    ) -> None:
        super().__init__()
        self.landmark_head = nn.ModuleList(
            [
                nn.Conv2d(
                    in_channels,
                    num_anchors * 10,
                    kernel_size=(1, 1),
                    stride=1,
                    padding=0,
                )
                for _ in range(fpn_num)
            ]
        )

    def forward(self, x: list[Tensor]) -> Tensor:
        output_tensors: list[Tensor] = []
        for feature, layer in zip(x, self.landmark_head):
            output_tensors.append(layer(feature).permute(0, 2, 3, 1).contiguous())

        landmarks = torch.cat(
            [out.view(out.shape[0], -1, 10) for out in output_tensors], dim=1
        )
        return landmarks

## Necks

In [ ]:
from __future__ import annotations

import torch.nn.functional as F
from torch import Tensor, nn



class FPN(nn.Module):
    """
    FPN (Feature Pyramid Network) for multi-scale feature map extraction and merging.
    Uses 1x1 convolutions for output layers and 3x3 convolutions for merging layers.
    """

    def __init__(self, in_channels_list: list[int], out_channels: int) -> None:
        """
        Initializes the FPN module.

        Args:
            in_channels_list (list of int): List of input channel sizes for each pyramid level.
            out_channels (int): Number of output channels for the feature pyramid.
        """
        super().__init__()
        leaky = 0.1 if out_channels <= 64 else 0
        # Define 1x1 convolution output layers
        self.output1 = Conv2dNormActivation(
            in_channels_list[0], out_channels, kernel_size=1, negative_slope=leaky
        )
        self.output2 = Conv2dNormActivation(
            in_channels_list[1], out_channels, kernel_size=1, negative_slope=leaky
        )
        self.output3 = Conv2dNormActivation(
            in_channels_list[2], out_channels, kernel_size=1, negative_slope=leaky
        )

        # Define merge layers using 3x3 convolutions
        self.merge1 = Conv2dNormActivation(
            out_channels, out_channels, kernel_size=3, negative_slope=leaky
        )
        self.merge2 = Conv2dNormActivation(
            out_channels, out_channels, kernel_size=3, negative_slope=leaky
        )

    def forward(self, inputs) -> list[Tensor]:
        """
        Forward pass of the FPN module.

        Args:
            inputs (dict or list): Input feature maps from different levels of the pyramid.

        Returns:
            list: List of merged output feature maps at different scales.
        """
        inputs = list(inputs.values())

        # Apply output layers to each feature map
        output1 = self.output1(inputs[0])
        output2 = self.output2(inputs[1])
        output3 = self.output3(inputs[2])

        # Merge outputs with upsampling and addition
        upsample3 = F.interpolate(output3, size=output2.shape[2:], mode="nearest")
        output2 = self.merge2(output2 + upsample3)

        upsample2 = F.interpolate(output2, size=output1.shape[2:], mode="nearest")
        output1 = self.merge1(output1 + upsample2)

        # Return merged feature maps
        return [output1, output2, output3]

In [ ]:
from __future__ import annotations
import torch
import torch.nn.functional as F
from torch import Tensor, nn

__all__ = ["SSH"]


class SSH(nn.Module):
    """
    SSH (Single Stage Headless) Module for feature extraction.
    Combines 3x3, 5x5, and 7x7 convolutions with batch normalization and optional LeakyReLU activations.
    """

    def __init__(self, in_channel: int, out_channels: int) -> None:
        """
        Initializes the SSH module.

        Args:
            in_channel (int): Number of input channels.
            out_channels (int): Number of output channels, must be divisible by 4.
        """
        super().__init__()

        if out_channels % 4 != 0:
            raise ValueError("out_channels must be divisible by 4.")
        leaky = 0.1 if out_channels <= 64 else 0

        # 3x3 Convolution branch
        self.conv3X3 = Conv2dNormActivation(
            in_channel, out_channels // 2, kernel_size=3, activation_layer=None
        )

        # 5x5 Convolution branch
        self.conv5X5_1 = Conv2dNormActivation(
            in_channel, out_channels // 4, kernel_size=3, negative_slope=leaky
        )
        self.conv5X5_2 = Conv2dNormActivation(
            out_channels // 4, out_channels // 4, kernel_size=3, activation_layer=None
        )

        # 7x7 Convolution branch
        self.conv7X7_2 = Conv2dNormActivation(
            out_channels // 4, out_channels // 4, kernel_size=3, negative_slope=leaky
        )
        self.conv7x7_3 = Conv2dNormActivation(
            out_channels // 4, out_channels // 4, kernel_size=3, activation_layer=None
        )

    def forward(self, x: Tensor) -> Tensor:
        """
        Forward pass of the SSH module.

        Args:
            x (Tensor): Input feature map.

        Returns:
            Tensor: Output feature map after applying SSH operations and ReLU activation.
        """
        conv3X3 = self.conv3X3(x)
        conv5X5_1 = self.conv5X5_1(x)
        conv5X5 = self.conv5X5_2(conv5X5_1)
        conv7X7 = self.conv7x7_3(self.conv7X7_2(conv5X5_1))

        out = torch.cat([conv3X3, conv5X5, conv7X7], dim=1)
        return F.relu(out, inplace=True)

## Builder

In [ ]:

BACKBONE_FACTORY = {
    "mobilenet_v2": mobilenet_v2,
}


def build_backbone(name, pretrained=False):
    if name not in BACKBONE_FACTORY:
        raise ValueError(f"Unknown backbone: {name}")

    return BACKBONE_FACTORY[name](pretrained=pretrained)


def get_layer_extractor(config, backbone):
    if config["name"] == "mobilenet_v2":
        return IntermediateLayerGetterByIndex(backbone, indexes=config["return_layers"])

    raise ValueError(f"Unsupported layer extractor for backbone: {config['name']}")

## Model

In [ ]:
from __future__ import annotations

import torch.nn.functional as F
from torch import Tensor, nn


def _resolve_fpn_in_channels(backbone: nn.Module, indexes: list[int]) -> list[int]:
    features = getattr(backbone, "features", None)
    if not isinstance(features, nn.Sequential):
        raise TypeError("backbone.features must be an nn.Sequential")

    channels: list[int] = []
    for idx in indexes:
        if idx < 0 or idx >= len(features):
            raise IndexError(
                f"Feature index {idx} is out of range for backbone.features"
            )

        feature_layer = features[idx]
        out_channels = getattr(feature_layer, "out_channels", None)
        if not isinstance(out_channels, int):
            raise TypeError(
                f"Feature layer at index {idx} does not expose an integer out_channels"
            )
        channels.append(out_channels)

    return channels


class RetinaFace(nn.Module):
    def __init__(self, cfg: dict | None = None) -> None:
        """
        RetinaFace model constructor.

        Args:
            cfg (dict): A configuration dictionary containing model parameters.
        """
        super().__init__()

        if cfg is None:
            raise ValueError("cfg must not be None.")

        backbone = build_backbone(cfg["name"], cfg["pretrain"])
        self.fx = get_layer_extractor(cfg, backbone)  # feature extraction

        num_anchors = 2
        out_channels = cfg["out_channel"]
        feature_indexes = cfg["return_layers"]
        fpn_in_channels = _resolve_fpn_in_channels(backbone, feature_indexes)
        self.fpn_in_channels = fpn_in_channels

        self.fpn = FPN(fpn_in_channels, out_channels)
        self.ssh1 = SSH(out_channels, out_channels)
        self.ssh2 = SSH(out_channels, out_channels)
        self.ssh3 = SSH(out_channels, out_channels)

        self.class_head = ClassHead(
            in_channels=cfg["out_channel"], num_anchors=num_anchors, fpn_num=3
        )
        self.bbox_head = BboxHead(
            in_channels=cfg["out_channel"], num_anchors=num_anchors, fpn_num=3
        )
        self.landmark_head = LandmarkHead(
            in_channels=cfg["out_channel"], num_anchors=num_anchors, fpn_num=3
        )

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        out = self.fx(x)
        fpn = self.fpn(out)

        # single-stage headless module
        feature1 = self.ssh1(fpn[0])
        feature2 = self.ssh2(fpn[1])
        feature3 = self.ssh3(fpn[2])

        features = [feature1, feature2, feature3]

        classifications = self.class_head(features)
        bbox_regressions = self.bbox_head(features)
        landmark_regressions = self.landmark_head(features)

        if self.training:
            output = (bbox_regressions, classifications, landmark_regressions)
        else:
            output = (
                bbox_regressions,
                F.softmax(classifications, dim=-1),
                landmark_regressions,
            )
        return output

# Train Utils

In [ ]:
from __future__ import annotations

import json
import random
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import torch
from torch import nn

__all__ = [
    "AverageMeter",
    "EpochMeters",
    "append_jsonl",
    "count_parameters",
    "ensure_dir",
    "format_seconds",
    "load_checkpoint",
    "save_checkpoint",
    "save_json",
    "set_seed",
    "timestamp_run_name",
]


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def timestamp_run_name(prefix: str = "run") -> str:
    return f"{prefix}_{time.strftime('%Y%m%d_%H%M%S')}"


def save_json(path: str | Path, payload: dict[str, Any]) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, sort_keys=True)
        f.write("\n")


def append_jsonl(path: str | Path, payload: dict[str, Any]) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    with path.open("a", encoding="utf-8") as f:
        json.dump(payload, f, sort_keys=True)
        f.write("\n")


def set_seed(seed: int | None) -> None:
    if seed is None:
        return

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def count_parameters(model: nn.Module, trainable_only: bool = True) -> int:
    params = model.parameters()
    if trainable_only:
        params = (param for param in params if param.requires_grad)
    return sum(param.numel() for param in params)


def format_seconds(seconds: float) -> str:
    total_seconds = max(int(seconds), 0)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours > 0:
        return f"{hours:d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


@dataclass
class AverageMeter:
    total: float = 0.0
    count: int = 0

    def update(self, value: float, n: int = 1) -> None:
        self.total += float(value) * int(n)
        self.count += int(n)

    @property
    def avg(self) -> float:
        if self.count == 0:
            return 0.0
        return self.total / self.count


@dataclass
class EpochMeters:
    loc: AverageMeter = field(default_factory=AverageMeter)
    conf: AverageMeter = field(default_factory=AverageMeter)
    landm: AverageMeter = field(default_factory=AverageMeter)
    total: AverageMeter = field(default_factory=AverageMeter)

    def update(
        self, loss_loc: float, loss_conf: float, loss_landm: float, total: float
    ) -> None:
        self.loc.update(loss_loc)
        self.conf.update(loss_conf)
        self.landm.update(loss_landm)
        self.total.update(total)

    def as_dict(self) -> dict[str, float]:
        return {
            "loss_loc": self.loc.avg,
            "loss_conf": self.conf.avg,
            "loss_landm": self.landm.avg,
            "loss_total": self.total.avg,
        }


def save_checkpoint(
    path: str | Path,
    *,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler | None,
    scaler: torch.GradScaler | None,
    epoch: int,
    global_step: int,
    best_loss: float,
    config: dict[str, Any],
    args: dict[str, Any],
    metrics: dict[str, Any] | None = None,
) -> Path:
    path = Path(path)
    ensure_dir(path.parent)

    payload = {
        "epoch": int(epoch),
        "global_step": int(global_step),
        "best_loss": float(best_loss),
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict() if scheduler is not None else None,
        "scaler": scaler.state_dict() if scaler is not None else None,
        "config": config,
        "args": args,
        "metrics": metrics or {},
    }
    torch.save(payload, path)
    return path


def load_checkpoint(
    path: str | Path,
    *,
    model: nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
    scheduler: torch.optim.lr_scheduler.LRScheduler | None = None,
    scaler: torch.GradScaler | None = None,
    device: torch.device | str = "cpu",
    strict: bool = True,
) -> dict[str, Any]:
    checkpoint = torch.load(Path(path), map_location=device)
    model.load_state_dict(checkpoint["model"], strict=strict)

    if optimizer is not None and checkpoint.get("optimizer") is not None:
        optimizer.load_state_dict(checkpoint["optimizer"])
    if scheduler is not None and checkpoint.get("scheduler") is not None:
        scheduler.load_state_dict(checkpoint["scheduler"])
    if scaler is not None and checkpoint.get("scaler") is not None:
        scaler.load_state_dict(checkpoint["scaler"])

    return checkpoint

# Train

In [ ]:
from __future__ import annotations

import argparse
import itertools
import math
import time
from contextlib import nullcontext
from pathlib import Path
from typing import Any, Callable, Sequence

import torch
from torch import Tensor
from torch.optim import SGD
from torch.optim.lr_scheduler import MultiStepLR
from torch.utils.data import DataLoader
from tqdm.auto import tqdm



def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Train RetinaFace on WIDER FACE train split"
    )
    parser.add_argument("--config", default="mobilenet_v2", help="config/backbone name")
    parser.add_argument(
        "--train-root", default="/kaggle/input/datasets/quanghijr/widerface/datasets/train", help="train split root"
    )
    parser.add_argument(
        "--annotation-file", default="/kaggle/input/datasets/quanghijr/widerface/datasets/train/train_annotations.txt", help="override annotation file path"
    )
    parser.add_argument("--image-root", default="/kaggle/input/datasets/quanghijr/widerface/datasets/train/images", help="override image root path")
    parser.add_argument(
        "--output-dir", default="/kaggle/working/outputs", help="directory for training runs"
    )
    parser.add_argument("--run-name", default=None, help="custom run directory name")
    parser.add_argument("--resume", default="/kaggle/working/outputs/retinaface_mobilenet_v2_20260401_140959/checkpoints/latest.pth", help="checkpoint to resume from")
    parser.add_argument(
        "--device",
        default="cuda" if torch.cuda.is_available() else "cpu",
        help="device, e.g. cuda, cuda:0, cpu",
    )
    parser.add_argument(
        "--workers", type=int, default=4, help="number of dataloader workers"
    )
    parser.add_argument(
        "--batch-size", type=int, default=None, help="override config batch size"
    )
    parser.add_argument(
        "--epochs", type=int, default=None, help="override config epochs"
    )
    parser.add_argument(
        "--image-size", type=int, default=None, help="override config image size"
    )
    parser.add_argument(
        "--min-face-size",
        type=float,
        default=0.0,
        help="minimum resized face size kept during random crop augmentation",
    )
    parser.add_argument(
        "--overlap-thresh",
        type=float,
        default=0.30,
        help="IoU threshold for positive anchor matching",
    )
    parser.add_argument(
        "--best-prior-iou-floor",
        type=float,
        default=0.10,
        help="minimum best-prior IoU required before force matching a GT",
    )
    parser.add_argument("--lr", type=float, default=1e-3, help="base learning rate")
    parser.add_argument("--momentum", type=float, default=0.9, help="SGD momentum")
    parser.add_argument("--weight-decay", type=float, default=5e-4, help="weight decay")
    parser.add_argument("--gamma", type=float, default=0.1, help="LR scheduler gamma")
    parser.add_argument(
        "--warmup-iters",
        type=int,
        default=1000,
        help="linear warmup iterations before using base LR",
    )
    parser.add_argument(
        "--save-every",
        type=int,
        default=5,
        help="save periodic checkpoints every N epochs",
    )
    parser.add_argument(
        "--log-interval", type=int, default=20, help="log every N optimizer steps"
    )
    parser.add_argument(
        "--clip-grad-norm",
        type=float,
        default=0.0,
        help="clip gradient norm if > 0",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=42,
        help="random seed for Python, NumPy, and PyTorch",
    )
    parser.add_argument(
        "--amp",
        action=argparse.BooleanOptionalAction,
        default=True,
        help="enable automatic mixed precision on CUDA",
    )
    parser.add_argument(
        "--pretrain",
        action=argparse.BooleanOptionalAction,
        default=None,
        help="override backbone pretrained weights usage",
    )
    parser.add_argument(
        "--limit-batches",
        type=int,
        default=0,
        help="for debugging: stop each epoch after this many batches (0 = full epoch)",
    )
    parser.add_argument(
        "--tqdm",
        action=argparse.BooleanOptionalAction,
        default=True,
        help="show tqdm progress bar during training",
    )
    args, _ = parser.parse_known_args()
    return args


def build_run_dir(args: argparse.Namespace) -> Path:
    if args.resume:
        resume_path = Path(args.resume).resolve()
        if resume_path.parent.name == "checkpoints":
            return resume_path.parent.parent
        return resume_path.parent

    run_name = args.run_name or timestamp_run_name(prefix=f"retinaface_{args.config}")
    return ensure_dir(Path(args.output_dir) / run_name)


def build_dataset(args: argparse.Namespace, cfg: dict[str, Any]) -> WiderFaceDataset:
    train_root = Path(args.train_root)
    annotation_file = (
        Path(args.annotation_file)
        if args.annotation_file
        else train_root / "train_annotations.txt"
    )
    image_root = Path(args.image_root) if args.image_root else train_root / "images"

    transform = RetinaFaceTrainTransform(
        image_size=int(cfg["image_size"]),
        min_face_size=float(args.min_face_size),
    )
    return WiderFaceDataset(
        annotation_file=annotation_file,
        image_root=image_root,
        transform=transform,
        normalize_targets=True,
        to_tensor=True,
    )


def build_dataloader(
    dataset: WiderFaceDataset,
    args: argparse.Namespace,
    device: torch.device,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.workers,
        collate_fn=widerface_collate,
        pin_memory=device.type == "cuda",
        persistent_workers=args.workers > 0,
    )


def resolve_device(device_arg: str) -> torch.device:
    device = torch.device(device_arg)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is not available")
    return device


def move_images_to_device(
    images: Tensor | Sequence[Tensor], device: torch.device
) -> Tensor:
    if isinstance(images, Tensor):
        return images.to(device=device, non_blocking=device.type == "cuda")

    if isinstance(images, Sequence) and all(
        isinstance(image, Tensor) for image in images
    ):
        image_shapes = {tuple(image.shape) for image in images}
        if len(image_shapes) != 1:
            raise ValueError(
                "images must have the same shape to be stacked for training"
            )
        batch = torch.stack(list(images), dim=0)
        return batch.to(device=device, non_blocking=device.type == "cuda")

    raise TypeError("batch images must be a Tensor or a sequence of Tensors")


def set_learning_rate(
    optimizer: torch.optim.Optimizer,
    base_lrs: list[float],
    scale: float,
) -> None:
    for group, base_lr in zip(optimizer.param_groups, base_lrs):
        group["lr"] = base_lr * scale


def build_grad_scaler(enabled: bool):
    return torch.GradScaler(enabled=enabled)


def train_one_epoch(
    *,
    model: RetinaFace,
    criterion: MultiBoxLoss,
    optimizer: torch.optim.Optimizer,
    scaler: Any,
    data_loader: DataLoader,
    priors: Tensor,
    device: torch.device,
    epoch: int,
    global_step: int,
    warmup_iters: int,
    base_lrs: list[float],
    log_interval: int,
    clip_grad_norm: float,
    limit_batches: int,
    use_amp: bool,
    show_progress: bool,
    log: Callable[..., None],
) -> tuple[EpochMeters, int, float]:
    model.train()
    criterion.train()

    meters = EpochMeters()
    start_time = time.time()
    num_batches = len(data_loader)
    effective_total = (
        min(num_batches, limit_batches) if limit_batches > 0 else num_batches
    )

    data_iterable = (
        itertools.islice(data_loader, limit_batches)
        if limit_batches > 0
        else data_loader
    )
    progress = tqdm(
        total=effective_total,
        desc=f"Epoch {epoch + 1}",
        dynamic_ncols=True,
        leave=False,
        disable=not show_progress,
    )

    try:
        for batch_idx, (images, targets) in enumerate(data_iterable):

            if warmup_iters > 0 and global_step < warmup_iters:
                warmup_scale = float(global_step + 1) / float(warmup_iters)
                set_learning_rate(optimizer, base_lrs, warmup_scale)

            images = move_images_to_device(images, device)
            optimizer.zero_grad(set_to_none=True)

            autocast_context = (
                torch.autocast(
                    device_type=device.type, dtype=torch.float16, enabled=True
                )
                if use_amp
                else nullcontext()
            )
            with autocast_context:
                predictions = model(images)
                loss_loc, loss_conf, loss_landm = criterion(
                    predictions, priors, targets
                )
                loss_total = criterion.combine_losses(loss_loc, loss_conf, loss_landm)

            scaler.scale(loss_total).backward()
            if clip_grad_norm > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), max_norm=clip_grad_norm
                )
            scaler.step(optimizer)
            scaler.update()

            global_step += 1

            meters.update(
                loss_loc=float(loss_loc.detach()),
                loss_conf=float(loss_conf.detach()),
                loss_landm=float(loss_landm.detach()),
                total=float(loss_total.detach()),
            )
            lr = optimizer.param_groups[0]["lr"]
            progress.update(1)
            progress.set_postfix(
                lr=f"{lr:.2e}",
                loss=f"{meters.total.avg:.4f}",
                loc=f"{meters.loc.avg:.4f}",
                conf=f"{meters.conf.avg:.4f}",
                landm=f"{meters.landm.avg:.4f}",
                refresh=batch_idx + 1 == effective_total,
            )

            if (
                batch_idx % max(log_interval, 1) == 0
                or batch_idx + 1 == effective_total
            ):
                elapsed = time.time() - start_time
                processed_batches = batch_idx + 1
                eta_seconds = (
                    elapsed
                    / max(processed_batches, 1)
                    * max(
                        effective_total - processed_batches,
                        0,
                    )
                )
                log(
                    f"epoch {epoch + 1} step {processed_batches}/{effective_total} "
                    f"lr={lr:.6f} "
                    f"loss={meters.total.avg:.4f} "
                    f"(loc={meters.loc.avg:.4f}, conf={meters.conf.avg:.4f}, landm={meters.landm.avg:.4f}) "
                    f"eta={format_seconds(eta_seconds)}",
                    echo=not show_progress,
                )
    finally:
        progress.close()

    epoch_time = time.time() - start_time
    return meters, global_step, epoch_time


def main() -> None:
    args = parse_args()
    run_dir = build_run_dir(args)
    checkpoints_dir = ensure_dir(run_dir / "checkpoints")
    metrics_path = run_dir / "metrics.jsonl"
    log_path = run_dir / "train.log"

    def log(message: str, *, echo: bool = True) -> None:
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{timestamp}] {message}"
        if echo:
            if args.tqdm:
                tqdm.write(line)
            else:
                print(line, flush=True)
        with log_path.open("a", encoding="utf-8") as f:
            f.write(line + "\n")

    set_seed(args.seed)
    device = resolve_device(args.device)
    torch.set_float32_matmul_precision("high")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    cfg = get_config(args.config)
    if args.batch_size is None:
        args.batch_size = int(cfg["batch_size"])
    if args.epochs is None:
        args.epochs = int(cfg["epochs"])
    if args.image_size is not None:
        cfg["image_size"] = int(args.image_size)
    if args.pretrain is not None:
        cfg["pretrain"] = bool(args.pretrain)

    dataset = build_dataset(args, cfg)
    data_loader = build_dataloader(dataset, args, device)

    model = RetinaFace(cfg).to(device)
    criterion = MultiBoxLoss(
        cfg,
        overlap_thresh=float(args.overlap_thresh),
        best_prior_iou_floor=float(args.best_prior_iou_floor),
    ).to(device)
    optimizer = SGD(
        model.parameters(),
        lr=args.lr,
        momentum=args.momentum,
        weight_decay=args.weight_decay,
    )
    scheduler = MultiStepLR(
        optimizer,
        milestones=[int(milestone) for milestone in cfg["milestones"]],
        gamma=args.gamma,
    )
    use_amp = bool(args.amp and device.type == "cuda")
    scaler = build_grad_scaler(use_amp)

    base_lrs = [group["lr"] for group in optimizer.param_groups]
    priors = (
        PriorBox(
            cfg,
            image_size=(int(cfg["image_size"]), int(cfg["image_size"])),
        )
        .forward()
        .to(device)
    )

    start_epoch = 0
    global_step = 0
    best_loss = math.inf
    current_epoch = start_epoch - 1

    if args.resume:
        checkpoint = load_checkpoint(
            args.resume,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            device=device,
        )
        start_epoch = int(checkpoint.get("epoch", -1)) + 1
        global_step = int(checkpoint.get("global_step", 0))
        best_loss = float(checkpoint.get("best_loss", math.inf))
        log(
            f"resumed from {args.resume} at epoch {start_epoch + 1} "
            f"(global_step={global_step}, best_loss={best_loss:.4f})"
        )

    save_json(run_dir / "config.json", cfg)
    save_json(run_dir / "args.json", vars(args))
    save_json(
        run_dir / "run_info.json",
        {
            "device": str(device),
            "dataset_size": len(dataset),
            "trainable_parameters": count_parameters(model),
            "num_priors": int(priors.size(0)),
            "use_amp": use_amp,
        },
    )

    log(
        f"starting training on {device} with {len(dataset)} samples, "
        f"batch_size={args.batch_size}, epochs={args.epochs}, "
        f"trainable_params={count_parameters(model):,}"
    )

    try:
        for epoch in range(start_epoch, args.epochs):
            current_epoch = epoch
            epoch_meters, global_step, epoch_time = train_one_epoch(
                model=model,
                criterion=criterion,
                optimizer=optimizer,
                scaler=scaler,
                data_loader=data_loader,
                priors=priors,
                device=device,
                epoch=epoch,
                global_step=global_step,
                warmup_iters=args.warmup_iters,
                base_lrs=base_lrs,
                log_interval=args.log_interval,
                clip_grad_norm=args.clip_grad_norm,
                limit_batches=args.limit_batches,
                use_amp=use_amp,
                show_progress=args.tqdm,
                log=log,
            )

            if global_step >= args.warmup_iters:
                scheduler.step()

            summary = {
                "epoch": epoch + 1,
                "global_step": global_step,
                "lr": optimizer.param_groups[0]["lr"],
                "epoch_time_sec": epoch_time,
                **epoch_meters.as_dict(),
            }
            append_jsonl(metrics_path, summary)
            save_json(run_dir / "latest_metrics.json", summary)

            if summary["loss_total"] < best_loss:
                best_loss = summary["loss_total"]

            latest_path = checkpoints_dir / "latest.pth"
            save_checkpoint(
                latest_path,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                epoch=epoch,
                global_step=global_step,
                best_loss=best_loss,
                config=cfg,
                args=vars(args),
                metrics=summary,
            )

            if summary["loss_total"] == best_loss:
                save_checkpoint(
                    checkpoints_dir / "best.pth",
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    scaler=scaler,
                    epoch=epoch,
                    global_step=global_step,
                    best_loss=best_loss,
                    config=cfg,
                    args=vars(args),
                    metrics=summary,
                )

            if args.save_every > 0 and (epoch + 1) % args.save_every == 0:
                save_checkpoint(
                    checkpoints_dir / f"epoch_{epoch + 1:03d}.pth",
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    scaler=scaler,
                    epoch=epoch,
                    global_step=global_step,
                    best_loss=best_loss,
                    config=cfg,
                    args=vars(args),
                    metrics=summary,
                )

            log(
                f"finished epoch {epoch + 1}/{args.epochs} in {format_seconds(epoch_time)} "
                f"loss={summary['loss_total']:.4f} "
                f"(loc={summary['loss_loc']:.4f}, conf={summary['loss_conf']:.4f}, "
                f"landm={summary['loss_landm']:.4f}) best={best_loss:.4f}"
            )

    except KeyboardInterrupt:
        interrupt_path = checkpoints_dir / "interrupt.pth"
        save_checkpoint(
            interrupt_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            epoch=max(current_epoch, 0),
            global_step=global_step,
            best_loss=best_loss,
            config=cfg,
            args=vars(args),
            metrics={"status": "interrupted"},
        )
        log(f"training interrupted, checkpoint saved to {interrupt_path}")
        raise


if __name__ == "__main__":
    main()